In [1]:
import numpy as np, pandas as pd, xarray as xr
from pathlib import Path

DAYS = {   # label: (Processed_Dataset folder, optional time window)
    "25.09 Pallas":      ("20260925_pallas_raw_P02_P07_CLARA", None),   # e.g. ("2026-09-25 00:54", "2026-09-25 11:12")
    "01.10 Pallas":      ("20261001_pallas_interpolation_CLARA", None),
    "02.10 Pallas":      ("20261002_pallas_interpolation_CLARA", None),
    "30.09.2023 Cabauw": ("20230930", None),
}
MAX_GAP = 3   # holes up to 3 gates (drizzle screen, neighbour filter) don't split a cloud

def main_layer(mask_1d, r, max_gap=MAX_GAP):
    """Base and top [m] of the thickest cloud layer in one profile, after bridging small holes."""
    idx = np.flatnonzero(mask_1d)
    if idx.size == 0:
        return np.nan, np.nan
    breaks = np.flatnonzero(np.diff(idx) > max_gap + 1)      # gaps larger than max_gap = new layer
    starts = np.r_[idx[0], idx[breaks + 1]]
    ends   = np.r_[idx[breaks], idx[-1]]
    k = np.argmax(r[ends] - r[starts])                       # thickest layer
    return r[starts[k]], r[ends[k]]

q = lambda x, p: np.nanpercentile(x, p)
rows = []
for day, (folder, window) in DAYS.items():
    ds = xr.open_dataset(Path(f"../data/processed/{folder}/retrieval_input_4km_alpha_kappa_calculated.nc"))
    if window:
        ds = ds.sel(Time=slice(*window))
    # cloud geometry from the detection mask BEFORE the drizzle screen (if the file has it)
    mask_name = "common_mask_raw" if "common_mask_raw" in ds else "common_mask"
    m = ds[mask_name].fillna(0).astype(bool).transpose("Time", "range").values   # fillna: NaN must not become True
    r = ds["range"].values
    base, top = np.array([main_layer(p, r) for p in m]).T
    cloudy = np.isfinite(base)
    thick = top - base
    lwp = ds["LWP"].values[cloudy]
    rows.append({
        "day": day,
        "mask used": mask_name,
        "cloudy profiles": f"{cloudy.sum()} ({100 * cloudy.mean():.0f} %)",
        "base at lowest gate (%)": f"{100 * np.mean(base[cloudy] <= r[0] + 1):.0f}",
        "base [m] median (p5-p95)": f"{q(base, 50):.0f} ({q(base, 5):.0f}-{q(base, 95):.0f})",
        "top [m] median (p5-p95)":  f"{q(top, 50):.0f} ({q(top, 5):.0f}-{q(top, 95):.0f})",
        "thickness [m] median (p5-p95)": f"{q(thick, 50):.0f} ({q(thick, 5):.0f}-{q(thick, 95):.0f})",
        "MWR LWP [g/m2] median (p5-p95)": f"{q(lwp, 50):.0f} ({q(lwp, 5):.0f}-{q(lwp, 95):.0f})",
        "MWR LWP min-max": f"{np.nanmin(lwp):.0f}-{np.nanmax(lwp):.0f}",
    })

table = pd.DataFrame(rows).set_index("day").T
print(table.to_string())

day                                25.09 Pallas     01.10 Pallas     02.10 Pallas 30.09.2023 Cabauw
mask used                       common_mask_raw  common_mask_raw  common_mask_raw       common_mask
cloudy profiles                   17229 (100 %)     22748 (75 %)     30697 (87 %)      12610 (85 %)
base at lowest gate (%)                      96               42               47                 0
base [m] median (p5-p95)          104 (104-104)    194 (104-358)    119 (104-656)  2033 (1242-2824)
top [m] median (p5-p95)           537 (358-686)    596 (328-745)    537 (373-999)  2372 (1431-3088)
thickness [m] median (p5-p95)     432 (253-566)     328 (89-641)     358 (75-835)      216 (75-509)
MWR LWP [g/m2] median (p5-p95)      94 (29-168)        29 (2-49)        35 (4-85)         31 (8-68)
MWR LWP min-max                          15-226            -3-61           -3-172            -1-107
